# Stat 220, Unit 4 Homework: Prediction and Choosing Predictors

`delivery_routes.csv` is 500 completed delivery routes. `minutes` is how long the
route took. The predictors that make sense are `stops`, `packages`, `miles`, `downtown`, and
`rain`. The company also records `van_age_years`, `dispatcher_rating`, and `month`.

```python
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score, train_test_split

routes = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/delivery_routes.csv")
folds = KFold(5, shuffle=True, random_state=0)
routes.head()
```

**Problem 1.** *Planning tomorrow's schedule.* Tomorrow's route has 45 stops, 110 packages,
38 miles, goes downtown, and no rain is forecast.

Part a. Report the predicted minutes, the confidence interval, and the prediction interval. Say in one sentence what each of the two intervals is about.

In [ ]:
fit = smf.ols("minutes ~ stops + packages + miles + downtown + rain", data=routes).fit()

new = pd.DataFrame({"stops": [45], "packages": [110], "miles": [38],
                    "downtown": [1], "rain": [0]})
print(fit.get_prediction(new).summary_frame(alpha=0.05).round(1))

_Your answer:_



Part b. The dispatcher has to tell one driver when to expect to finish tomorrow. Which of the two intervals should they use, and why?

_Your answer:_



Part c. The operations manager asks a different question: across all the downtown routes like this one, what is our average time? Which interval answers that one?

_Your answer:_



**Problem 2.** *A route that looks ordinary.* A planner asks for a prediction for a route with
25 stops and 120 packages.

Part a. Each value is common on its own. Report how many completed routes are near each value, and how many are near both.

In [ ]:
near_stops = routes.stops.between(20, 30)
near_packages = routes.packages.between(110, 130)

print("routes near this stop count   :", near_stops.sum())
print("routes near this package count:", near_packages.sum())
print("routes near both              :", (near_stops & near_packages).sum())
print("correlation between stops and packages:", round(routes.stops.corr(routes.packages), 2))

_Your answer:_



Part b. In three or four sentences, explain what your part a numbers mean for a prediction for this route, and what you would tell the planner who asked for it.

_Your answer:_



**Problem 3.** *How wrong will it be?* A new depot has 50 completed routes so far, and the
manager wants one number for how far off a time estimate typically is.

Part a. Report both numbers, and say which one you would give the manager.

In [ ]:
depot = routes.sample(50, random_state=3)
X = depot[["stops", "packages", "miles", "downtown", "rain"]]
y = depot["minutes"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
m = LinearRegression().fit(X_train, y_train)

print("on the rows it was fitted on:", round(np.sqrt(((y_train - m.predict(X_train))**2).mean()), 1))
print("on the rows held back       :", round(np.sqrt(((y_test - m.predict(X_test))**2).mean()), 1))

_Your answer:_



Part b. Explain in two or three sentences why those two numbers differ, and which way the difference would go if the depot had 500 routes instead of 50.

_Your answer:_



**Problem 4.** *Which predictors belong?* Still the 50-route depot.

Part a. Report the cross-validated error, the $R^2$, and the AIC for each set. Say which set you would ship.

In [ ]:
sets = {
    "stops only": ["stops"],
    "the five that make sense": ["stops", "packages", "miles", "downtown", "rain"],
    "those five plus three junk": ["stops", "packages", "miles", "downtown", "rain",
                                   "van_age_years", "dispatcher_rating", "month"],
}
for name, cols in sets.items():
    mse = -cross_val_score(LinearRegression(), depot[cols], depot.minutes, cv=folds,
                           scoring="neg_mean_squared_error").mean()
    f = smf.ols("minutes ~ " + " + ".join(cols), data=depot).fit()
    print(f"{name:<28} CV error {np.sqrt(mse):5.1f}   R2 {f.rsquared:.4f}   AIC {f.aic:.1f}")

_Your answer:_



Part b. One set has the highest $R^2$ and is still not the set you would ship. Explain what $R^2$ is doing here in two or three sentences.

_Your answer:_



Part c. A colleague wants to add a squared term in `stops` on top of the five. Say what you would check before agreeing, and what you would expect with only 50 routes.

_Your answer:_



**Problem 5.** *What can and cannot be said.* No computer.

Part a. A colleague ran stepwise selection over 65 columns, kept the 11 with p-values under 0.05, and reports an $R^2$ of 0.96 with every predictor significant. Write the two or three sentences you would say in response.

_Your answer:_



Part b. A lasso fit sends `rain` to exactly zero. Does that show rain has no effect on how long a route takes? Explain.

_Your answer:_



Part c. The model predicts well and `packages` has a large coefficient. The manager asks whether splitting the same deliveries into more packages would make routes take longer. Answer in two or three sentences.

_Your answer:_

